In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# criação dos nomes/caminhos dos schemas de cada camada
catalog = "vcredit_medallion"
silver_schema = f"{catalog}.silver"
gold_schema = f"{catalog}.gold"


In [0]:
tables = spark.sql(f"SHOW TABLES IN {silver_schema}")
display(tables)

##gold.dim_motivo

In [0]:
#gold.dim_motivo (origem silver.tb_motivo)

# partição única explícita: a dimensão é pequena, então juntar tudo numa partição não pesa
w_sk = Window.partitionBy(F.lit(1)).orderBy(F.col("id_motivo").cast("int"))

df_silver_motivo = spark.read.table(f"{silver_schema}.base_motivos")

df_dim_motivo = (
    df_silver_motivo
    .withColumn("SK_motivo", F.row_number().over(w_sk).cast("bigint"))
    .select(
        F.col("SK_motivo"),
        F.col("id_motivo").cast("int").alias("id_motivo"),
        F.col("nome_motivo").cast("string").alias("nome"),
        F.col("categoria").cast("string").alias("categoria"),
        F.col("criticidade").cast("string").alias("criticidade"),
    )
    .orderBy("SK_motivo")
)

display(df_dim_motivo)

df_dim_motivo.write.mode("overwrite").saveAsTable(f"{gold_schema}.dim_motivo")